# Multilingual Knowledge Extraction & Exploration Assistant

AI-only interview prototype for a scanned, multilingual, handwritten textbook. The notebook is organized in the same order as the pipeline: Module 1 first, then chunking and retrieval on top of the cleaned structure.

In [ ]:
from pathlib import Path
import sys


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists():
            return candidate
    return start


project_root = _find_project_root(Path.cwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Project root set to: {project_root}')

## Module 1: Document Understanding + Preprocessing

The goal here is not just OCR text dumping. The code tries to recover a readable document hierarchy: headings, paragraphs, bullets, and page breaks. For the demo, I use a text-only fallback so the notebook stays runnable even when OCR libraries are unavailable.

In [ ]:
from src.document_understanding import DocumentProcessor, OCRConfig

sample_text = '''
Chapter 1: परिचय
Definitions
This textbook explains multilingual knowledge extraction.
- Point 1: Clean the OCR text.
- Point 2: Preserve headings and bullet points.

વિષયવસ્તુ
આ પુસ્તકમાં Gujarati, English, અને Sanskrit mixed content છે.
- મહત્વપૂર્ણ વિચાર 1
- મહત્વપૂર્ણ વિચાર 2
'''.strip()

processor = DocumentProcessor(OCRConfig(backend='text'))
document = processor.process_text(sample_text, source_name='sample_textbook')

print(document.markdown)
print()
for block in document.blocks:
    print(block.block_type, '|', block.page_number, '|', block.text)

### Expected output shape

```markdown
# Chapter 1: परिचय

## Definitions
This textbook explains multilingual knowledge extraction.
- Point 1: Clean the OCR text.
- Point 2: Preserve headings and bullet points.

## વિષયવસ્તુ
આ પુસ્તકમાં Gujarati, English, અને Sanskrit mixed content છે.
- મહત્વપૂર્ણ વિચાર 1
- મહત્વપૂર્ણ વિચાર 2
```

## Module 2: Chunking + Embedding

The chunker keeps headings as anchors and splits only when a token budget is exceeded. This is the simplest approach that still works well for interview-scale retrieval systems.

In [ ]:
from src.chunking import StructureAwareChunker, ChunkConfig
from src.retrieval import ChunkRetriever

chunker = StructureAwareChunker(ChunkConfig(max_tokens=60, overlap_tokens=15))
chunks = chunker.chunk_markdown(document.markdown, source_name=document.source_name)

print('Chunks:')
for chunk in chunks:
    print(chunk.chunk_id, chunk.section_path, '->', chunk.text)

retriever = ChunkRetriever()
retriever.fit(chunks)

## Module 3: Question Answering / Exploration

The retrieval layer combines semantic search with a conservative extractive answer. If you want to plug in an LLM later, the interface already supports that through an optional client callback.

In [ ]:
query = 'How does the system clean OCR text?'
keyword = retriever.keyword_search(query)
semantic = retriever.answer(query)

print('Keyword hits:')
for score, chunk in zip(keyword.scores, keyword.retrieved_chunks):
    print(score, chunk.chunk_id)

print()
print('Answer:')
print(semantic.answer)

## Interview talking points

- I prioritized explainable heuristics first, because OCR noise and mixed scripts are easier to debug that way.
- I made the modules independent, so OCR, chunking, and retrieval can be swapped out later without rewriting the whole pipeline.
- I kept a text-only fallback in the notebook so the prototype can be demonstrated even when OCR engines are not installed locally.